<a href="https://colab.research.google.com/github/kavisaranithangavel-dev/Creation-of-Security-Operations-Dashboard-for-Threat-Detection-with-Risk-Mitigation-Analytics/blob/main/SOC_SQL_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Practice — Security Operations
Hands-on SQL practice using SQLite and a SOC database (analysts, assets, alerts, login events, blocked IPs).


# SQL for Security Operations

Create a SOC database with four tables: analysts, assets, alerts and login_events. Use primary keys, foreign keys, and SQLite-compatible SQL.

In [ ]:
import sqlite3

conn = sqlite3.connect("soc.db")
conn.execute("PRAGMA foreign_keys = ON")
cursor = conn.cursor()

# Start clean so the notebook can be re-run
for t in ["login_events", "alerts", "assets", "analysts"]:
    cursor.execute(f"DROP TABLE IF EXISTS {t}")

cursor.execute("""
CREATE TABLE analysts (
    analyst_id INTEGER PRIMARY KEY,
    analyst_name TEXT NOT NULL,
    tier TEXT,
    shift TEXT
)
""")

cursor.execute("""
CREATE TABLE assets (
    asset_id INTEGER PRIMARY KEY,
    hostname TEXT NOT NULL,
    asset_type TEXT,
    criticality TEXT,
    owner_dept TEXT
)
""")

cursor.execute("""
CREATE TABLE alerts (
    alert_id INTEGER PRIMARY KEY,
    asset_id INTEGER,
    analyst_id INTEGER,
    alert_type TEXT,
    mitre_tactic TEXT,
    severity TEXT,
    status TEXT DEFAULT 'New',
    created_at TEXT,
    resolved_at TEXT,
    FOREIGN KEY (asset_id) REFERENCES assets(asset_id),
    FOREIGN KEY (analyst_id) REFERENCES analysts(analyst_id)
)
""")

cursor.execute("""
CREATE TABLE login_events (
    event_id INTEGER PRIMARY KEY,
    username TEXT,
    source_ip TEXT,
    country TEXT,
    result TEXT,
    event_time TEXT
)
""")

conn.commit()
print("SOC tables created!")

SOC tables created!


Insert sample records into all four tables.

In [ ]:
cursor.executemany("INSERT INTO analysts VALUES (?, ?, ?, ?)", [
    (1, 'Meera',  'L1', 'Day'),
    (2, 'Karthik','L2', 'Night'),
    (3, 'Divya',  'L3', 'Day'),
    (4, 'Suresh', 'L1', 'Night'),
])

cursor.executemany("INSERT INTO assets VALUES (?, ?, ?, ?, ?)", [
    (10, 'web-prod-01',  'Server',      'Critical', 'IT'),
    (11, 'db-prod-01',   'Database',    'Critical', 'IT'),
    (12, 'hr-laptop-07', 'Laptop',      'Medium',   'HR'),
    (13, 'dev-vm-03',    'VM',          'Low',      'Engineering'),
    (14, 'vpn-gw-01',    'Firewall',    'High',     'Network'),
])

cursor.executemany("INSERT INTO alerts VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)", [
    (1, 10, 1, 'Brute Force',        'Credential Access',   'High',     'Closed',        '2026-11-01 02:10:00', '2026-11-01 03:00:00'),
    (2, 11, 3, 'SQL Injection',      'Initial Access',      'Critical', 'Investigating', '2026-11-01 09:30:00', None),
    (3, 12, 1, 'Phishing Email',     'Initial Access',      'Medium',   'Closed',        '2026-11-02 10:00:00', '2026-11-02 11:30:00'),
    (4, 13, 4, 'Port Scan',          'Discovery',           'Low',      'Closed',        '2026-11-02 23:15:00', '2026-11-03 00:05:00'),
    (5, 10, 2, 'Malware Detected',   'Execution',           'Critical', 'Open',          '2026-11-03 01:45:00', None),
    (6, 14, 2, 'Impossible Travel',  'Initial Access',      'High',     'Open',          '2026-11-03 04:20:00', None),
    (7, 12, 1, 'Phishing Email',     'Initial Access',      'High',     'Closed',        '2026-11-04 08:50:00', '2026-11-04 10:10:00'),
    (8, 11, 3, 'Data Exfiltration',  'Exfiltration',        'Critical', 'Investigating', '2026-11-05 14:05:00', None),
    (9, 10, 4, 'Brute Force',        'Credential Access',   'Medium',   'Closed',        '2026-11-05 22:40:00', '2026-11-06 00:10:00'),
])

cursor.executemany("INSERT INTO login_events VALUES (?, ?, ?, ?, ?, ?)", [
    (1,  'arun',  '203.0.113.10', 'IN', 'FAILED',  '2026-11-05 22:01:00'),
    (2,  'arun',  '203.0.113.10', 'IN', 'FAILED',  '2026-11-05 22:01:20'),
    (3,  'arun',  '203.0.113.10', 'IN', 'FAILED',  '2026-11-05 22:01:45'),
    (4,  'arun',  '203.0.113.10', 'IN', 'SUCCESS', '2026-11-05 22:02:10'),
    (5,  'rahul', '198.51.100.7', 'RU', 'FAILED',  '2026-11-05 23:15:00'),
    (6,  'rahul', '198.51.100.7', 'RU', 'FAILED',  '2026-11-05 23:15:10'),
    (7,  'anu',   '192.0.2.44',   'IN', 'SUCCESS', '2026-11-06 09:00:00'),
    (8,  'anu',   '198.51.100.99','CN', 'SUCCESS', '2026-11-06 09:20:00'),
    (9,  'admin', '198.51.100.7', 'RU', 'FAILED',  '2026-11-06 01:00:00'),
    (10, 'admin', '198.51.100.7', 'RU', 'FAILED',  '2026-11-06 01:00:05'),
    (11, 'admin', '198.51.100.7', 'RU', 'FAILED',  '2026-11-06 01:00:09'),
    (12, 'admin', '198.51.100.7', 'RU', 'FAILED',  '2026-11-06 01:00:14'),
])
conn.commit()
print("Sample data inserted!")

Sample data inserted!


Display all records from every table.

In [ ]:
for table in ["analysts", "assets", "alerts", "login_events"]:
    print(f"\n{table.upper()}")
    for row in cursor.execute(f"SELECT * FROM {table}"):
        print(row)


ANALYSTS
(1, 'Meera', 'L1', 'Day')
(2, 'Karthik', 'L2', 'Night')
(3, 'Divya', 'L3', 'Day')
(4, 'Suresh', 'L1', 'Night')

ASSETS
(10, 'web-prod-01', 'Server', 'Critical', 'IT')
(11, 'db-prod-01', 'Database', 'Critical', 'IT')
(12, 'hr-laptop-07', 'Laptop', 'Medium', 'HR')
(13, 'dev-vm-03', 'VM', 'Low', 'Engineering')
(14, 'vpn-gw-01', 'Firewall', 'High', 'Network')

ALERTS
(1, 10, 1, 'Brute Force', 'Credential Access', 'High', 'Closed', '2026-11-01 02:10:00', '2026-11-01 03:00:00')
(2, 11, 3, 'SQL Injection', 'Initial Access', 'Critical', 'Investigating', '2026-11-01 09:30:00', None)
(3, 12, 1, 'Phishing Email', 'Initial Access', 'Medium', 'Closed', '2026-11-02 10:00:00', '2026-11-02 11:30:00')
(4, 13, 4, 'Port Scan', 'Discovery', 'Low', 'Closed', '2026-11-02 23:15:00', '2026-11-03 00:05:00')
(5, 10, 2, 'Malware Detected', 'Execution', 'Critical', 'Open', '2026-11-03 01:45:00', None)
(6, 14, 2, 'Impossible Travel', 'Initial Access', 'High', 'Open', '2026-11-03 04:20:00', None)
(7, 12, 

### 1.1 Filtering

Retrieve all alerts whose status is Open.

In [ ]:
for row in cursor.execute("SELECT * FROM alerts WHERE status = 'Open'"):
    print(row)

(5, 10, 2, 'Malware Detected', 'Execution', 'Critical', 'Open', '2026-11-03 01:45:00', None)
(6, 14, 2, 'Impossible Travel', 'Initial Access', 'High', 'Open', '2026-11-03 04:20:00', None)


Find Critical alerts that are not yet closed (AND / !=).

In [ ]:
for row in cursor.execute("""
SELECT alert_id, alert_type, status
FROM alerts
WHERE severity = 'Critical' AND status != 'Closed'
"""):
    print(row)

(2, 'SQL Injection', 'Investigating')
(5, 'Malware Detected', 'Open')
(8, 'Data Exfiltration', 'Investigating')


Find alerts with severity High or Critical using IN.

In [ ]:
for row in cursor.execute("SELECT alert_id, alert_type, severity FROM alerts WHERE severity IN ('High','Critical')"):
    print(row)

(1, 'Brute Force', 'High')
(2, 'SQL Injection', 'Critical')
(5, 'Malware Detected', 'Critical')
(6, 'Impossible Travel', 'High')
(7, 'Phishing Email', 'High')
(8, 'Data Exfiltration', 'Critical')


Find alerts created between 2 Nov and 4 Nov 2026 using BETWEEN.

In [ ]:
for row in cursor.execute("""
SELECT alert_id, alert_type, created_at
FROM alerts
WHERE created_at BETWEEN '2026-11-02 00:00:00' AND '2026-11-04 23:59:59'
"""):
    print(row)

(3, 'Phishing Email', '2026-11-02 10:00:00')
(4, 'Port Scan', '2026-11-02 23:15:00')
(5, 'Malware Detected', '2026-11-03 01:45:00')
(6, 'Impossible Travel', '2026-11-03 04:20:00')
(7, 'Phishing Email', '2026-11-04 08:50:00')


Find all alert types that contain the word 'Phishing' using LIKE.

In [ ]:
for row in cursor.execute("SELECT alert_id, alert_type FROM alerts WHERE alert_type LIKE '%Phishing%'"):
    print(row)

(3, 'Phishing Email')
(7, 'Phishing Email')


Find alerts that are still unresolved (resolved_at IS NULL).

In [ ]:
for row in cursor.execute("SELECT alert_id, alert_type, status FROM alerts WHERE resolved_at IS NULL"):
    print(row)

(2, 'SQL Injection', 'Investigating')
(5, 'Malware Detected', 'Open')
(6, 'Impossible Travel', 'Open')
(8, 'Data Exfiltration', 'Investigating')


List the unique MITRE ATT&CK tactics seen in alerts.

In [ ]:
for row in cursor.execute("SELECT DISTINCT mitre_tactic FROM alerts"):
    print(row)

('Credential Access',)
('Initial Access',)
('Discovery',)
('Execution',)
('Exfiltration',)


### 1.2 Sorting and aggregation

Show alerts sorted by created_at, newest first.

In [ ]:
for row in cursor.execute("SELECT alert_id, alert_type, created_at FROM alerts ORDER BY created_at DESC"):
    print(row)

(9, 'Brute Force', '2026-11-05 22:40:00')
(8, 'Data Exfiltration', '2026-11-05 14:05:00')
(7, 'Phishing Email', '2026-11-04 08:50:00')
(6, 'Impossible Travel', '2026-11-03 04:20:00')
(5, 'Malware Detected', '2026-11-03 01:45:00')
(4, 'Port Scan', '2026-11-02 23:15:00')
(3, 'Phishing Email', '2026-11-02 10:00:00')
(2, 'SQL Injection', '2026-11-01 09:30:00')
(1, 'Brute Force', '2026-11-01 02:10:00')


Count alerts for each alert_type, highest count first.

In [ ]:
for row in cursor.execute("""
SELECT alert_type, COUNT(*) AS total
FROM alerts
GROUP BY alert_type
ORDER BY total DESC
"""):
    print(row)

('Phishing Email', 2)
('Brute Force', 2)
('SQL Injection', 1)
('Port Scan', 1)
('Malware Detected', 1)
('Impossible Travel', 1)
('Data Exfiltration', 1)


Count alerts per severity level.

In [ ]:
for row in cursor.execute("SELECT severity, COUNT(*) FROM alerts GROUP BY severity"):
    print(row)

('Critical', 3)
('High', 3)
('Low', 1)
('Medium', 2)


Find alert types that occurred more than once (GROUP BY + HAVING).

In [ ]:
for row in cursor.execute("""
SELECT alert_type, COUNT(*)
FROM alerts
GROUP BY alert_type
HAVING COUNT(*) > 1
"""):
    print(row)

('Brute Force', 2)
('Phishing Email', 2)


Show the total number of alerts and the number that are still unresolved.

In [ ]:
cursor.execute("""
SELECT COUNT(*) AS total_alerts,
       SUM(CASE WHEN resolved_at IS NULL THEN 1 ELSE 0 END) AS unresolved
FROM alerts
""")
print(cursor.fetchone())

(9, 4)


### 1.3 Joins

Show each alert with the hostname of the affected asset (INNER JOIN).

In [ ]:
for row in cursor.execute("""
SELECT a.alert_id, s.hostname, a.alert_type, a.severity
FROM alerts a
JOIN assets s ON a.asset_id = s.asset_id
"""):
    print(row)

(1, 'web-prod-01', 'Brute Force', 'High')
(2, 'db-prod-01', 'SQL Injection', 'Critical')
(3, 'hr-laptop-07', 'Phishing Email', 'Medium')
(4, 'dev-vm-03', 'Port Scan', 'Low')
(5, 'web-prod-01', 'Malware Detected', 'Critical')
(6, 'vpn-gw-01', 'Impossible Travel', 'High')
(7, 'hr-laptop-07', 'Phishing Email', 'High')
(8, 'db-prod-01', 'Data Exfiltration', 'Critical')
(9, 'web-prod-01', 'Brute Force', 'Medium')


Show each alert with the name of the analyst handling it.

In [ ]:
for row in cursor.execute("""
SELECT a.alert_id, n.analyst_name, n.tier, a.alert_type
FROM alerts a
JOIN analysts n ON a.analyst_id = n.analyst_id
"""):
    print(row)

(1, 'Meera', 'L1', 'Brute Force')
(2, 'Divya', 'L3', 'SQL Injection')
(3, 'Meera', 'L1', 'Phishing Email')
(4, 'Suresh', 'L1', 'Port Scan')
(5, 'Karthik', 'L2', 'Malware Detected')
(6, 'Karthik', 'L2', 'Impossible Travel')
(7, 'Meera', 'L1', 'Phishing Email')
(8, 'Divya', 'L3', 'Data Exfiltration')
(9, 'Suresh', 'L1', 'Brute Force')


Show all assets and their alerts, including assets that have **no** alerts (LEFT JOIN).

In [ ]:
for row in cursor.execute("""
SELECT s.hostname, a.alert_id, a.alert_type
FROM assets s
LEFT JOIN alerts a ON s.asset_id = a.asset_id
"""):
    print(row)

('web-prod-01', 1, 'Brute Force')
('web-prod-01', 9, 'Brute Force')
('web-prod-01', 5, 'Malware Detected')
('db-prod-01', 8, 'Data Exfiltration')
('db-prod-01', 2, 'SQL Injection')
('hr-laptop-07', 3, 'Phishing Email')
('hr-laptop-07', 7, 'Phishing Email')
('dev-vm-03', 4, 'Port Scan')
('vpn-gw-01', 6, 'Impossible Travel')


Three-table join: analyst name, hostname, alert type and severity for all Critical alerts.

In [ ]:
for row in cursor.execute("""
SELECT n.analyst_name, s.hostname, a.alert_type, a.severity
FROM alerts a
JOIN analysts n ON a.analyst_id = n.analyst_id
JOIN assets   s ON a.asset_id   = s.asset_id
WHERE a.severity = 'Critical'
"""):
    print(row)

('Divya', 'db-prod-01', 'SQL Injection', 'Critical')
('Karthik', 'web-prod-01', 'Malware Detected', 'Critical')
('Divya', 'db-prod-01', 'Data Exfiltration', 'Critical')


Count alerts per asset criticality level.

In [ ]:
for row in cursor.execute("""
SELECT s.criticality, COUNT(*) AS alert_count
FROM alerts a
JOIN assets s ON a.asset_id = s.asset_id
GROUP BY s.criticality
ORDER BY alert_count DESC
"""):
    print(row)

('Critical', 5)
('Medium', 2)
('Low', 1)
('High', 1)


Analyst workload: how many alerts is each analyst handling?

In [ ]:
for row in cursor.execute("""
SELECT n.analyst_name, COUNT(a.alert_id) AS alerts_handled
FROM analysts n
LEFT JOIN alerts a ON n.analyst_id = a.analyst_id
GROUP BY n.analyst_name
ORDER BY alerts_handled DESC
"""):
    print(row)

('Meera', 3)
('Suresh', 2)
('Karthik', 2)
('Divya', 2)


### 1.4 Subqueries and CASE

Find hostnames that have at least one Critical alert (subquery with IN).

In [ ]:
for row in cursor.execute("""
SELECT hostname
FROM assets
WHERE asset_id IN (SELECT asset_id FROM alerts WHERE severity = 'Critical')
"""):
    print(row)

('web-prod-01',)
('db-prod-01',)


Find assets that have never had an alert (NOT IN).

In [ ]:
for row in cursor.execute("""
SELECT hostname
FROM assets
WHERE asset_id NOT IN (SELECT asset_id FROM alerts)
"""):
    print(row)

Assign a response priority to each alert using CASE WHEN: Critical = P1, High = P2, Medium = P3, otherwise P4.

In [ ]:
for row in cursor.execute("""
SELECT alert_id, alert_type, severity,
       CASE severity
            WHEN 'Critical' THEN 'P1'
            WHEN 'High'     THEN 'P2'
            WHEN 'Medium'   THEN 'P3'
            ELSE 'P4'
       END AS priority
FROM alerts
"""):
    print(row)

(1, 'Brute Force', 'High', 'P2')
(2, 'SQL Injection', 'Critical', 'P1')
(3, 'Phishing Email', 'Medium', 'P3')
(4, 'Port Scan', 'Low', 'P4')
(5, 'Malware Detected', 'Critical', 'P1')
(6, 'Impossible Travel', 'High', 'P2')
(7, 'Phishing Email', 'High', 'P2')
(8, 'Data Exfiltration', 'Critical', 'P1')
(9, 'Brute Force', 'Medium', 'P3')


### 1.5 Security analytics with dates

Calculate resolution time in minutes for every closed alert (MTTR per alert).

In [ ]:
for row in cursor.execute("""
SELECT alert_id, alert_type,
       ROUND((julianday(resolved_at) - julianday(created_at)) * 24 * 60) AS minutes_to_resolve
FROM alerts
WHERE resolved_at IS NOT NULL
"""):
    print(row)

(1, 'Brute Force', 50.0)
(3, 'Phishing Email', 90.0)
(4, 'Port Scan', 50.0)
(7, 'Phishing Email', 80.0)
(9, 'Brute Force', 90.0)


Calculate the Mean Time To Resolve (MTTR) in minutes across all closed alerts.

In [ ]:
cursor.execute("""
SELECT ROUND(AVG((julianday(resolved_at) - julianday(created_at)) * 24 * 60), 1) AS mttr_minutes
FROM alerts
WHERE resolved_at IS NOT NULL
""")
print("MTTR (minutes):", cursor.fetchone()[0])

MTTR (minutes): 72.0


Count alerts raised per day.

In [ ]:
for row in cursor.execute("""
SELECT DATE(created_at) AS day, COUNT(*) AS alerts
FROM alerts
GROUP BY day
ORDER BY day
"""):
    print(row)

('2026-11-01', 2)
('2026-11-02', 2)
('2026-11-03', 2)
('2026-11-04', 1)
('2026-11-05', 2)


Find alerts raised during night hours (22:00–05:59).

In [ ]:
for row in cursor.execute("""
SELECT alert_id, alert_type, created_at
FROM alerts
WHERE CAST(strftime('%H', created_at) AS INTEGER) >= 22
   OR CAST(strftime('%H', created_at) AS INTEGER) < 6
"""):
    print(row)

(1, 'Brute Force', '2026-11-01 02:10:00')
(4, 'Port Scan', '2026-11-02 23:15:00')
(5, 'Malware Detected', '2026-11-03 01:45:00')
(6, 'Impossible Travel', '2026-11-03 04:20:00')
(9, 'Brute Force', '2026-11-05 22:40:00')


SLA check: list closed alerts that took longer than 60 minutes to resolve.

In [ ]:
for row in cursor.execute("""
SELECT alert_id, alert_type, severity,
       ROUND((julianday(resolved_at) - julianday(created_at)) * 24 * 60) AS minutes
FROM alerts
WHERE resolved_at IS NOT NULL
  AND (julianday(resolved_at) - julianday(created_at)) * 24 * 60 > 60
"""):
    print(row)

(3, 'Phishing Email', 'Medium', 90.0)
(7, 'Phishing Email', 'High', 80.0)
(9, 'Brute Force', 'Medium', 90.0)


### 1.6 Threat hunting on login events

Count failed logins per username and source IP.

In [ ]:
for row in cursor.execute("""
SELECT username, source_ip, COUNT(*) AS failures
FROM login_events
WHERE result = 'FAILED'
GROUP BY username, source_ip
ORDER BY failures DESC
"""):
    print(row)

('admin', '198.51.100.7', 4)
('arun', '203.0.113.10', 3)
('rahul', '198.51.100.7', 2)


Brute-force detection: find source IPs with 3 or more failed logins.

In [ ]:
for row in cursor.execute("""
SELECT source_ip, COUNT(*) AS failures
FROM login_events
WHERE result = 'FAILED'
GROUP BY source_ip
HAVING COUNT(*) >= 3
"""):
    print(row)

('198.51.100.7', 6)
('203.0.113.10', 3)


Possible compromised account: users with failed logins followed by a SUCCESS from the same IP.

In [ ]:
for row in cursor.execute("""
SELECT DISTINCT f.username, f.source_ip
FROM login_events f
JOIN login_events s
  ON f.username = s.username
 AND f.source_ip = s.source_ip
 AND s.result = 'SUCCESS'
 AND s.event_time > f.event_time
WHERE f.result = 'FAILED'
"""):
    print(row)

('arun', '203.0.113.10')


Impossible-travel hint: users who logged in successfully from more than one country.

In [ ]:
for row in cursor.execute("""
SELECT username, COUNT(DISTINCT country) AS countries
FROM login_events
WHERE result = 'SUCCESS'
GROUP BY username
HAVING COUNT(DISTINCT country) > 1
"""):
    print(row)

('anu', 2)


Window function: rank source IPs by number of failed logins.

In [ ]:
for row in cursor.execute("""
SELECT source_ip, failures,
       RANK() OVER (ORDER BY failures DESC) AS rnk
FROM (
    SELECT source_ip, COUNT(*) AS failures
    FROM login_events
    WHERE result = 'FAILED'
    GROUP BY source_ip
)
"""):
    print(row)

('198.51.100.7', 6, 1)
('203.0.113.10', 3, 2)


### 1.7 Updating and constraints

Assign alert 5 to analyst 4 and change its status to Investigating (UPDATE).

In [ ]:
cursor.execute("UPDATE alerts SET analyst_id = 4, status = 'Investigating' WHERE alert_id = 5")
conn.commit()
print(cursor.execute("SELECT * FROM alerts WHERE alert_id = 5").fetchone())

(5, 10, 4, 'Malware Detected', 'Execution', 'Critical', 'Investigating', '2026-11-03 01:45:00', None)


Close alert 6 and record the resolution time.

In [ ]:
cursor.execute("UPDATE alerts SET status = 'Closed', resolved_at = '2026-11-03 06:00:00' WHERE alert_id = 6")
conn.commit()
print(cursor.execute("SELECT * FROM alerts WHERE alert_id = 6").fetchone())

(6, 14, 2, 'Impossible Travel', 'Initial Access', 'High', 'Closed', '2026-11-03 04:20:00', '2026-11-03 06:00:00')


Add a new column `ticket_id` to the alerts table (ALTER TABLE).

In [ ]:
cursor.execute("ALTER TABLE alerts ADD COLUMN ticket_id TEXT")
conn.commit()
print([c[1] for c in cursor.execute("PRAGMA table_info(alerts)")])

['alert_id', 'asset_id', 'analyst_id', 'alert_type', 'mitre_tactic', 'severity', 'status', 'created_at', 'resolved_at', 'ticket_id']


Create a table of blocked IPs (an IOC blocklist) with a UNIQUE constraint and a CHECK constraint on confidence.

In [ ]:
cursor.execute("DROP TABLE IF EXISTS blocked_ips")
cursor.execute("""
CREATE TABLE blocked_ips (
    ip TEXT PRIMARY KEY,
    reason TEXT NOT NULL,
    confidence INTEGER CHECK (confidence BETWEEN 0 AND 100),
    added_on TEXT DEFAULT CURRENT_TIMESTAMP
)
""")
cursor.executemany("INSERT INTO blocked_ips (ip, reason, confidence) VALUES (?, ?, ?)", [
    ('198.51.100.7',  'Repeated brute force', 95),
    ('198.51.100.99', 'Known malicious ASN',  70),
])
conn.commit()
for row in cursor.execute("SELECT * FROM blocked_ips"):
    print(row)

('198.51.100.7', 'Repeated brute force', 95, '2026-11-06 14:58:02')
('198.51.100.99', 'Known malicious ASN', 70, '2026-11-06 14:58:02')


Use the blocklist: find all login events that came from a blocked IP.

In [ ]:
for row in cursor.execute("""
SELECT l.username, l.source_ip, l.result, b.reason
FROM login_events l
JOIN blocked_ips b ON l.source_ip = b.ip
ORDER BY l.event_time
"""):
    print(row)

('rahul', '198.51.100.7', 'FAILED', 'Repeated brute force')
('rahul', '198.51.100.7', 'FAILED', 'Repeated brute force')
('admin', '198.51.100.7', 'FAILED', 'Repeated brute force')
('admin', '198.51.100.7', 'FAILED', 'Repeated brute force')
('admin', '198.51.100.7', 'FAILED', 'Repeated brute force')
('admin', '198.51.100.7', 'FAILED', 'Repeated brute force')
('anu', '198.51.100.99', 'SUCCESS', 'Known malicious ASN')


Constraint test: inserting a confidence of 150 must fail. (The error is intentional.)

In [ ]:
try:
    cursor.execute("INSERT INTO blocked_ips (ip, reason, confidence) VALUES ('10.0.0.1', 'test', 150)")
    conn.commit()
except sqlite3.IntegrityError as e:
    print("Rejected as expected:", e)

Rejected as expected: CHECK constraint failed: confidence BETWEEN 0 AND 100


Delete a false-positive alert (alert 4, a harmless internal port scan).

In [ ]:
cursor.execute("DELETE FROM alerts WHERE alert_id = 4")
conn.commit()
print("Remaining alerts:", cursor.execute("SELECT COUNT(*) FROM alerts").fetchone()[0])

Remaining alerts: 8
